In [1]:
### import libraries

### working with dates
import pandas as pd

### working with geodataframes
import geopandas as gpd

### opening raster files (GeoTIFFs)
import rioxarray as rxr

### combining rasters into a time series
import xarray as xr

### making interactive plots of vector and raster data
import hvplot.pandas
import hvplot.xarray

### downloading NDVI data with the AppEEARS API
import earthpy
import earthpy.api.appeears as eaapp

In [2]:
### open the rectangle around the 2022 Uljin wildfire that I drew on geojson.io
uljin_box_gdf = gpd.read_file('../data/uljin_box.geojson')

### make a site map of the study area on top of satellite images
hvplot.extension('bokeh')
site_map = uljin_box_gdf.hvplot(
    geo = True, tiles = 'EsriImagery',
    title = 'Study area around the 2022 Uljin wildfire',
    fill_color = None, line_color = 'white',
    frame_height = 350)
site_map

c:\Users\chwj1\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [3]:
### make a home for the data
project = earthpy.project.Project(dirname = 'uljin-ndvi')

### make descriptive download key for the data you'll download
download_key = 'uljin-ndvi-july-2019-2025-v2'

### initialize AppeearsDownloader for MODIS NDVI data
ndvi_downloader = eaapp.AppeearsDownloader(

    ### add download key
    download_key = download_key,

    ### define folder where data will download to
    project = project,

    ### specify remote sensing product
    product = 'MOD13Q1.061',

    ### and specific layer
    layer = '_250m_16_days_NDVI',

    ### only July, when plants are the greenest
    start_date = '07-01',
    end_date = '07-31',

    ### get the same July dates every year
    recurring = True,

    ### 2019-2021 is before the fire (March 2022), 2022-2024 is after,
    ### and 2025 shows more recovery
    year_range = [2019, 2025],

    ### set polygon for area of interest
    polygon = uljin_box_gdf
)

### download the prepared data (can be slow)
ndvi_downloader.download_files(cache = True)

<generator object Path.rglob at 0x000001D9E4A46680>

In [4]:
### find all the NDVI files (not the Quality files) and sort them by date
ndvi_paths = sorted(list(project.project_dir.rglob('*NDVI*.tif')))

### the date (like 2001193) starts 25 characters from the end of the file name
### and stops just before the 18th character from the end
doy_start = -25
doy_end = -18

### empty list to collect each NDVI image
ndvi_das = []

### go through the NDVI files one by one
for ndvi_path in ndvi_paths:

    ### cut the year and day of year out of the file name
    doy = ndvi_path.name[doy_start:doy_end]

    ### change it into a real date (%Y = year, %j = day of year)
    date = pd.to_datetime(doy, format = '%Y%j')

    ### open the tif file, change no-data values to NaN, and remove the band dimension
    da = rxr.open_rasterio(ndvi_path, mask_and_scale = True).squeeze()

    ### add the date to the image and make it a dimension
    da = da.assign_coords({'date': date})
    da = da.expand_dims({'date': 1})

    ### call the values 'NDVI' and add this image to the list
    da.name = 'NDVI'
    ndvi_das.append(da)

### stack all the NDVI images into one dataset along the date dimension
ndvi_da = xr.combine_by_coords(ndvi_das, coords = ['date'], compat = 'no_conflicts')

### check the result
ndvi_da

<xarray.Dataset> Size: 2MB
Dimensions:      (date: 21, y: 130, x: 169)
Coordinates:
  * date         (date) datetime64[us] 168B 2019-06-26 2019-07-12 ... 2025-07-28
  * y            (y) float64 1kB 37.22 37.22 37.22 37.21 ... 36.96 36.95 36.95
  * x            (x) float64 1kB 129.1 129.1 129.1 129.1 ... 129.4 129.4 129.5
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    NDVI         (date, y, x) float32 2MB 0.8721 0.9071 0.9159 ... nan nan nan

In [9]:
### calculate the NDVI before the fire
ndvi_pre = (

    ### start from the NDVI time series
    ndvi_da

        ### keep only 2019 to 2021 (before the fire in March 2022)
        .sel(date = slice('2019', '2021'))

        ### average the NDVI of each pixel over those dates
        .mean('date')

        ### get the NDVI values
        .NDVI
)

### calculate the NDVI after the fire
ndvi_post = (

    ### start from the NDVI time series
    ndvi_da

        ### keep only 2022 to 2024 (after the fire)
        .sel(date = slice('2022', '2024'))

        ### average the NDVI of each pixel over those dates
        .mean('date')

        ### get the NDVI values
        .NDVI
)

### calculate the change in NDVI
### after minus before (above 0 = more plants, below 0 = fewer plants)
ndvi_diff = ndvi_post - ndvi_pre

### open the burned area that I drew from the NDVI drop
uljin_burn_gdf = gpd.read_file('../data/uljin_burn.geojson')

### plot the difference
hvplot.extension('bokeh')
ndvi_change_map = (
    ndvi_diff.hvplot(

        ### use the x and y coordinates (longitude and latitude)
        x = 'x', y = 'y',

        ### pink = NDVI went down, green = NDVI went up
        cmap = 'PiYG',

        ### draw it as a map with lat/lon coordinates
        geo = True,

        ### add a title and labels
        title = 'Change in July NDVI after the 2022 Uljin wildfire',
        xlabel = 'Longitude', ylabel = 'Latitude',
        clabel = 'NDVI change (2022-2024 minus 2019-2021)',
        frame_height = 300,

        ### set the color range so the fire area is easier to see
        clim = (-0.3, 0.3))
    *

    ### draw the burned area on top of the NDVI map, with a legend
    uljin_burn_gdf.hvplot(
        geo = True, fill_color = None, line_color = 'black',
        label = 'Burned area')

### put the legend in the empty sea area at the top right of the map
).opts(legend_position = 'top_right')
ndvi_change_map

:Overlay
   .Image.I              :Image   [x,y]   (NDVI)
   .Polygons.Burned_area :Polygons   [Longitude,Latitude]

In [6]:
### make the area outside the burned area (the rest of the study rectangle)
unburned_gdf = uljin_box_gdf.overlay(uljin_burn_gdf, how = 'difference')

### clip the NDVI data to inside and outside the burned area
ndvi_burn_da = ndvi_da.rio.clip(uljin_burn_gdf.geometry, from_disk = True)
ndvi_unburn_da = ndvi_da.rio.clip(unburned_gdf.geometry, from_disk = True)

### calculate mean July NDVI inside the burned area for each year
ndvi_burn_df = (
    ndvi_burn_da

        ### put the images from the same year into one group
        .groupby(ndvi_burn_da.date.dt.year)

        ### average over all pixels and all dates in each year
        .mean(...)

        ### get the NDVI values and change them to a pandas DataFrame
        .NDVI.to_dataframe()
)

### calculate mean July NDVI outside the burned area for each year
ndvi_unburn_df = (
    ndvi_unburn_da
        .groupby(ndvi_unburn_da.date.dt.year)
        .mean(...)
        .NDVI.to_dataframe()
)

### put the two NDVI columns side by side with different names
ndvi_df = (
    ndvi_burn_df[['NDVI']]
    .join(
        ndvi_unburn_df[['NDVI']],
        lsuffix = ' burned', rsuffix = ' unburned')
)

### plot both lines on a single plot
hvplot.extension('bokeh')
ndvi_line_plot = ndvi_df.hvplot(
    title = 'Mean July NDVI in burned and unburned areas',
    xlabel = 'Year', ylabel = 'Mean July NDVI',
    frame_width = 400,

    ### leave some space below the lines for the legend
    ylim = (0.5, 0.9),

    ### put the legend at the bottom right and call it 'Area'
    legend = 'bottom_right',
    group_label = 'Area')
ndvi_line_plot

:NdOverlay   [Area]
   :Curve   [year]   (value)

In [7]:
### calculate the difference: burned minus unburned
### (below 0 = the burned area has less vegetation than the area around it)
ndvi_df['difference'] = ndvi_df['NDVI burned'] - ndvi_df['NDVI unburned']

### plot the difference
hvplot.extension('bokeh')
ndvi_diff_plot = ndvi_df['difference'].hvplot(
    title = 'Difference in mean July NDVI (burned minus unburned)',
    xlabel = 'Year', ylabel = 'NDVI difference',
    frame_width = 400)
ndvi_diff_plot

:Curve   [year]   (difference)

In [10]:
### save the plots as .html files in the img folder of the portfolio repository
### (the notebook runs from the notebooks folder, so go up one level)
hvplot.save(ndvi_change_map, '../img/uljin-ndvi-change-map.html')
hvplot.save(ndvi_line_plot, '../img/uljin-ndvi-burned-unburned.html')
hvplot.save(ndvi_diff_plot, '../img/uljin-ndvi-difference.html')
hvplot.save(site_map, '../img/uljin-site-map.html')